In [14]:
from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import random
import re
import sys
import time
import urllib.error
import urllib.request
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import date
from typing import Optional

# Prevent Cloudflare Error 1010 by setting standard browser User-Agent
_opener = urllib.request.build_opener()
_opener.addheaders = [("User-Agent", "Mozilla/5.0 (Windows NT 10.0; Win64; x64)")]
urllib.request.install_opener(_opener)


def load_btp_env():
    """Climbs directory tree to find .env in BTP root or checks standard locations."""
    candidates = [
        os.path.join(os.path.dirname(__file__), ".env") if "__file__" in globals() else None,
        "/Users/kartikgoyal/Desktop/BTP/.env",
        os.path.expanduser("~/Desktop/BTP/.env"),
        os.path.abspath(".env"),
        os.path.abspath("../.env"),
        os.path.abspath("../../.env"),
    ]
    for p in candidates:
        if p and os.path.exists(p):
            try:
                with open(p) as f:
                    for line in f:
                        line = line.strip()
                        if line.startswith("#") or "=" not in line:
                            continue
                        k, v = line.split("=", 1)
                        k, v = k.strip(), v.strip().strip("'").strip('"')
                        if k and v and k not in os.environ:
                            os.environ[k] = v
            except Exception:
                pass
    return None

load_btp_env()


In [3]:
SEED = 42               # Global random seed for corpus generation & experimental reproducibility
N_PROJECTS = 12
QUERY_DATES = [date(2026, 3, 1), date(2026, 6, 1)]
REVISION_DATE = date(2026, 5, 1)
TOOL_BUDGET = 12          # B
MAX_LLM_CALLS = 15
SEARCH_COST, READ_COST, FINISH_COST = 1, 2, 0
TOP_K = 5
FAIL_PENALTY = 10
ABSTAIN = "INSUFFICIENT EVIDENCE"
ATTRS = ["leader", "department", "budget_inr", "approval_status"]
STATUSES = ["APPROVED", "PENDING", "REJECTED"]
ATTR_SOURCES = {
    "leader": {"REG"}, "department": {"REG"},
    "budget_inr": {"BUD_INIT", "BUD_REV"},
    "approval_status": {"APP_INIT", "APP_REV"},
}

In [4]:
def d2s(d: date) -> str:
    return d.isoformat()
 
 
def s2d(s: str) -> date:
    return date.fromisoformat(s)

In [5]:
ADJ = ["Aurora", "Helios", "Borealis", "Cobalt", "Zephyr", "Obsidian", "Meridian", "Solstice",
       "Vertex", "Quasar", "Tidal", "Ember"]
NOUN = ["Grid", "Lattice", "Beacon", "Atlas", "Nexus", "Forge", "Canopy", "Relay"]
FIRST = ["Ananya", "Rohan", "Meera", "Karthik", "Isha", "Vikram", "Neha", "Arjun", "Divya",
         "Sanjay", "Pooja", "Rahul", "Kavya", "Manish", "Tara", "Nikhil"]
LAST = ["Rao", "Iyer", "Sharma", "Banerjee", "Patel", "Nair", "Gupta", "Mehta", "Reddy", "Joshi"]
DEPTS = ["Computer Science", "Electrical Engineering", "Mechanical Engineering",
         "Applied Mathematics", "Chemical Engineering", "Data Science"]
FILLER = [
    "This record was prepared by the Research Finance Office and filed in the institutional repository.",
    "Questions regarding this record should be directed to the Office of Sponsored Research.",
    "The contents are subject to the standard institutional review cycle.",
    "Please retain this document for audit purposes.",
]

@dataclass(frozen=True)
class Document:
    doc_id: str
    title: str
    issue_date: str
    effective_date: Optional[str]
    status: str            # ISSUED | DRAFT
    content: str
 
    def public(self) -> dict:
        return asdict(self)
 
 
class HiddenRecords:
    """Ground truth. NEVER passed to the agent / prompt; used only after termination."""
 
    def __init__(self):
        self.projects = {}      # pid -> dict of true values
        self.doc_meta = {}      # doc_id -> (pid, doc_type)
        self.doc_values = {}    # doc_id -> {attr: value stated in that doc}
 
    def truth(self, pid: str, d: date) -> dict:
        p = self.projects[pid]
        rev = d >= REVISION_DATE
        return {"leader": p["leader"], "department": p["department"],
                "budget_inr": p["budget_rev"] if rev else p["budget_init"],
                "approval_status": p["approval_rev"] if rev else p["approval_init"]}
 
    def source_docs(self, pid: str, d: date) -> dict:
        rev = d >= REVISION_DATE
        ids = self.projects[pid]["doc_ids"]
        return {"leader": ids["REG"], "department": ids["REG"],
                "budget_inr": ids["BUD_REV" if rev else "BUD_INIT"],
                "approval_status": ids["APP_REV" if rev else "APP_INIT"]}
 
    def memo_values(self, pid: str) -> dict:
        return self.doc_values[self.projects[pid]["doc_ids"]["MEMO"]]
 
 
def build_corpus(seed: Optional[int] = None):
    s = SEED if seed is None else seed
    rng = random.Random(s)
    names = rng.sample([f"{a} {n}" for a in ADJ for n in NOUN], N_PROJECTS)
    people = rng.sample([f"{f} {l}" for f in FIRST for l in LAST], 2 * N_PROJECTS)
    ids = [f"DOC-{i:03d}" for i in range(1, 6 * N_PROJECTS + 1)]
    rng.shuffle(ids)                      # doc ids carry no information about project/type
    id_iter = iter(ids)
    docs, hidden = {}, HiddenRecords()
 
    def fmt(d: date) -> str:
        return f"{d.day} {d.strftime('%B %Y')}"
 
    def filler():
        return rng.choice(FILLER)
 
    for i in range(N_PROJECTS):
        pid, name = f"P{i + 1:02d}", names[i]
        leader, memo_leader = people[i], people[N_PROJECTS + i]
        dept = rng.choice(DEPTS)
        memo_dept = rng.choice([x for x in DEPTS if x != dept])
        b1 = rng.randrange(5, 40) * 100_000
        b2 = b1 + rng.choice([-1, 1]) * rng.randrange(1, 10) * 50_000
        bm = b2 + rng.randrange(11, 20) * 50_000          # distinct from b1 and b2
        s1 = rng.choice(STATUSES)
        s2 = rng.choice([s for s in STATUSES if s != s1])
        sm = [s for s in STATUSES if s not in (s1, s2)][0]  # memo always wrong at either date
        tag = f"{name} ({pid})"
 
        reg_issue = date(2026, 1, rng.randint(5, 20))
        b1_issue = date(2026, 1, rng.randint(21, 31))
        a1_issue = date(2026, 1, rng.randint(21, 31))
        b2_issue = date(2026, 4, rng.randint(5, 25))
        a2_issue = date(2026, 4, rng.randint(5, 25))
        memo_issue = date(2026, rng.choice([3, 5, 6]), rng.randint(2, 25))  # can look "newest"
        eff1, eff2 = date(2026, 2, 1), date(2026, 5, 1)
 
        specs = [
            ("REG", f"Project Registration: {tag}", reg_issue, None, "ISSUED",
             f"Registration record for research project {tag}. The designated project leader is {leader}, "
             f"and the project is housed in the Department of {dept}. {filler()}",
             {"leader": leader, "department": dept}),
            ("BUD_INIT", f"Initial Budget Authorization: {tag}", b1_issue, eff1, "ISSUED",
             f"Budget authorization for project {tag}, effective {fmt(eff1)}. The authorized budget for the "
             f"project is INR {b1:,}. This is the initial authorization. {filler()}",
             {"budget_inr": b1}),
            ("BUD_REV", f"Revised Budget Authorization: {tag}", b2_issue, eff2, "ISSUED",
             f"Revised budget authorization for project {tag}, effective {fmt(eff2)}. This revision replaces the "
             f"initial budget authorization. The authorized budget for the project is INR {b2:,}. {filler()}",
             {"budget_inr": b2}),
            ("APP_INIT", f"Initial Approval Status Notice: {tag}", a1_issue, eff1, "ISSUED",
             f"Approval status notice for project {tag}, effective {fmt(eff1)}. The approval status of the "
             f"project is {s1}. This is the initial notice. {filler()}",
             {"approval_status": s1}),
            ("APP_REV", f"Revised Approval Status Notice: {tag}", a2_issue, eff2, "ISSUED",
             f"Revised approval status notice for project {tag}, effective {fmt(eff2)}. This revision replaces "
             f"the initial notice. The approval status of the project is {s2}. {filler()}",
             {"approval_status": s2}),
            ("MEMO", f"Draft Memorandum: {tag}", memo_issue, None, "DRAFT",
             f"DRAFT memorandum, not for circulation. Working notes on project {tag}. The designated project "
             f"leader is {memo_leader}, and the project is housed in the Department of {memo_dept}. The budget "
             f"under discussion is INR {bm:,}. The approval status of the project is {sm}, subject to "
             f"confirmation. {filler()}",
             {"leader": memo_leader, "department": memo_dept, "budget_inr": bm, "approval_status": sm}),
        ]
        doc_ids = {}
        for dtype, title, issue, eff, status, content, values in specs:
            did = next(id_iter)
            docs[did] = Document(did, title, d2s(issue), d2s(eff) if eff else None, status, content)
            hidden.doc_meta[did] = (pid, dtype)
            hidden.doc_values[did] = values
            doc_ids[dtype] = did
        hidden.projects[pid] = {"name": name, "leader": leader, "department": dept,
                                "budget_init": b1, "budget_rev": b2,
                                "approval_init": s1, "approval_rev": s2, "doc_ids": doc_ids}
    return docs, hidden
 
 
def make_question(pid: str, name: str, d: date) -> str:
    return (f"As of {d2s(d)}, who is responsible for project {name} ({pid}), which department does the "
            f"project belong to, what is its authorized budget, and what is its approval status?")

In [6]:
STOPWORDS = {"as", "of", "who", "is", "for", "the", "which", "does", "to", "what", "its", "and", "a", "an",
             "in", "on", "project", "belong", "responsible"}
 
 
def tokenize(s: str):
    return re.findall(r"[a-z0-9]+", s.lower())
 
 
class Retriever:
    """Plain TF-IDF keyword retrieval, implemented from scratch (shared by ALL methods).
 
    score(q, d) = (1 / sqrt(|d|)) * sum_{t in q ∩ d} (1 + ln tf(t,d)) * idf(t),   idf(t) = ln((N+1)/(df(t)+1))
    Stopwords (question boilerplate) are dropped from both queries and documents; the 1/sqrt(|d|) term is a
    length normalisation so long drafts do not win merely by containing more words.
    """
 
    def __init__(self, docs: dict):
        self.tf, df = {}, Counter()
        for did, d in docs.items():
            toks = [w for w in tokenize(d.title + " " + d.content) if w not in STOPWORDS]
            self.tf[did] = Counter(toks)
            df.update(set(toks))
        n = len(docs)
        self.idf = {t: max(math.log((n + 1) / (c + 1)), 0.0) for t, c in df.items()}
        self.norm = {did: math.sqrt(sum(tf.values())) for did, tf in self.tf.items()}
 
    def rank(self, query: str, k: int):
        q = {w for w in tokenize(query) if w not in STOPWORDS}
        scored = []
        for did, tf in self.tf.items():
            s = sum((1 + math.log(tf[t])) * self.idf.get(t, 0.0) for t in q if t in tf) / self.norm[did]
            if s > 0:
                scored.append((-s, did))
        scored.sort()
        return [did for _, did in scored[:k]]
 
 
class ToolEnv:
    """The ONLY window the agent has onto the corpus. Enforces access rules (defence in depth)."""
 
    def __init__(self, docs: dict, retriever: Retriever):
        self._docs, self._ret = docs, retriever
        self.returned_ids, self.read_ids = set(), []
 
    def search(self, query: str, k: int = TOP_K):
        ids = self._ret.rank(query, min(k, TOP_K))
        self.returned_ids.update(ids)
        return [{"doc_id": i, "title": self._docs[i].title, "issue_date": self._docs[i].issue_date,
                 "status": self._docs[i].status} for i in ids]
 
    def read(self, doc_id: str) -> dict:
        if doc_id not in self.returned_ids:
            raise PermissionError(f"{doc_id} was never returned by search")
        if doc_id in self.read_ids:
            raise PermissionError(f"{doc_id} already read")
        self.read_ids.append(doc_id)
        return self._docs[doc_id].public()

In [7]:
def classify_title(title: str) -> Optional[str]:
    """Agent-side document-type inference from the TITLE only (an observation, not hidden info)."""
    t = title.lower()
    if t.startswith("project registration"):
        return "REG"
    if t.startswith("draft memorandum"):
        return "MEMO"
    rev = t.startswith("revised")
    if "budget authorization" in t:
        return "BUD_REV" if rev else "BUD_INIT"
    if "approval status notice" in t:
        return "APP_REV" if rev else "APP_INIT"
    return None
 
 
_LD = re.compile(r"project leader is (.+?), and the project is housed in the Department of ([^.]+)\.")
_BUD = re.compile(r"INR ([\d,]+)")
_APP = re.compile(r"approval status of the project is (APPROVED|PENDING|REJECTED)")
 
 
def extract_values(content: str) -> dict:
    """Deterministic rule-based fact extraction from natural-language content."""
    out = {}
    m = _LD.search(content)
    if m:
        out["leader"], out["department"] = m.group(1).strip(), m.group(2).strip()
    m = _BUD.search(content)
    if m:
        out["budget_inr"] = int(m.group(1).replace(",", ""))
    m = _APP.search(content)
    if m:
        out["approval_status"] = m.group(1)
    return out
 
 
@dataclass
class Fact:
    attribute: str
    value: object
    doc_id: str
    doc_type: Optional[str]
    doc_status: str
    issue_date: str
    applicable_from: str     # effective date, or issue date for docs with no effective date
 
 
class EvidenceRecord:
    """Facts extracted ONLY from documents actually read, with provenance."""
 
    def __init__(self):
        self.facts: list[Fact] = []
 
    def add_document(self, doc: dict):
        dtype = classify_title(doc["title"])
        vals = extract_values(doc["content"])
        allowed = set(ATTRS) if dtype in (None, "MEMO") else {a for a, s in ATTR_SOURCES.items() if dtype in s}
        for attr, v in vals.items():
            if attr in allowed:
                self.facts.append(Fact(attr, v, doc["doc_id"], dtype, doc["status"], doc["issue_date"],
                                       doc["effective_date"] or doc["issue_date"]))
 
    def _eligible(self, d: date):
        return [f for f in self.facts if f.doc_status == "ISSUED" and s2d(f.applicable_from) <= d]
 
    def resolve(self, d: date) -> dict:
        """attribute -> best Fact (latest applicable ISSUED), or None."""
        best = {a: None for a in ATTRS}
        for f in self._eligible(d):
            cur = best[f.attribute]
            if cur is None or (f.applicable_from, f.issue_date) > (cur.applicable_from, cur.issue_date):
                best[f.attribute] = f
        return best
 
    def annotated(self, d: date):
        best = self.resolve(d)
        rows = []
        for f in self.facts:
            if f.doc_status != "ISSUED":
                standing = "IGNORE_draft"
            elif s2d(f.applicable_from) > d:
                standing = "NOT_YET_EFFECTIVE"
            elif best[f.attribute] is f:
                standing = "CURRENT"
            else:
                standing = "SUPERSEDED"
            rows.append({"attribute": f.attribute, "value": f.value, "doc_id": f.doc_id,
                         "doc_type": f.doc_type, "doc_status": f.doc_status,
                         "effective": f.applicable_from, "standing_on_query_date": standing})
        return rows

In [8]:
class EpisodeState:
    def __init__(self, question: str, pid: str, d: date, budget: int = TOOL_BUDGET):
        self.question, self.pid, self.query_date = question, pid, d
        self.budget_remaining, self.total_cost = budget, 0
        self.n_search = self.n_read = self.llm_calls = 0
        self.seen: dict[str, dict] = {}       # doc_id -> {title, issue_date, status, read}
        self.search_log: list[dict] = []      # [{query, ids}]
        self.read_docs: dict[str, dict] = {}  # full content of docs read (kept for RAG baseline / last obs)
        self.evidence = EvidenceRecord()
        self.history: list[dict] = []         # compact action log shown to the LLM
        self.last_observation = None
        self.invalid: list[dict] = []         # every rejected proposal
        self.read_ids: list[str] = []
        self.llm_facts: list[dict] = []       # facts the LLM recorded AND the controller verified as faithful
        self.fact_feedback: list[str] = []    # rejected-fact messages shown on the next prompt

In [9]:
RULES = """Interpretation rules (deterministic):
1. Only documents with status ISSUED are authoritative. DRAFT documents must never determine the answer.
2. A document becomes applicable on its effective date (a registration: on its issue date). A document whose effective date is AFTER the query date is not applicable.
3. For each attribute, the latest applicable ISSUED document determines its value.
4. Leader and department come from the project registration; budget from budget authorizations; approval status from approval-status notices."""
 
SCHEMA = """Reply with ONLY one JSON object, no other text:
{"action": "search", "args": {"query": "<text>", "k": 5}, "justification": "<one sentence>"}
{"action": "read", "args": {"doc_id": "<id returned by an earlier search>"}, "justification": "<one sentence>"}
{"action": "finish", "args": {"answer": <ANSWER>}, "justification": "<one sentence>"}
where <ANSWER> is either the string "INSUFFICIENT EVIDENCE" or
{"leader": "...", "department": "...", "budget_inr": <integer>, "approval_status": "APPROVED|PENDING|REJECTED",
 "support": {"leader": "<doc_id>", "department": "<doc_id>", "budget_inr": "<doc_id>", "approval_status": "<doc_id>"}}
Every doc_id in "support" must be a document you have actually read."""
 
SCHEMA_X = SCHEMA.replace("Reply with ONLY one JSON object, no other text:",
    "Reply with ONLY one JSON object, no other text. Besides action/args/justification it may carry an optional\n"
    "\"memory_update\": a list of facts you extracted from documents you have READ, each\n"
    "{\"attribute\": \"leader|department|budget_inr|approval_status\", \"value\": <exact value as written in the document>,\n"
    " \"doc_id\": \"<document it came from>\", \"note\": \"<your judgement: e.g. current / superseded / not yet effective / draft>\"}.\n"
    "Record every attribute value you find in each newly read document, including from drafts (mark them as such in the note).\n"
    "A recorded value must literally appear in the cited document; unfaithful entries are rejected and reported back to you.\n"
    "Format:")
 
 
def attribute_status(state: EpisodeState) -> dict:
    best = state.evidence.resolve(state.query_date)
    out = {}
    for a in ATTRS:
        unread = [{"doc_id": i, "title": m["title"], "issue_date": m["issue_date"], "status": m["status"]}
                  for i, m in state.seen.items()
                  if not m["read"] and classify_title(m["title"]) in ATTR_SOURCES[a]
                  and f"({state.pid})" in m["title"]]
        b = best[a]
        out[a] = {"best": None if b is None else {"value": b.value, "doc_id": b.doc_id,
                                                  "effective": b.applicable_from},
                  "unread_candidate_docs": unread}
    return out
 
 
def build_agent_prompt(state: EpisodeState, feedback: list[str], show_resolution: bool = True,
                       llm_extracts: bool = True) -> str:
    seen = [{"doc_id": i, **{k: m[k] for k in ("title", "issue_date", "status", "read")}}
            for i, m in state.seen.items()]
    parts = [
        "You are the decision policy of an evidence-grounded research agent. Choose ONE next action.",
        RULES,
        f"QUESTION: {state.question}",
        f"QUERY_DATE: {d2s(state.query_date)}",
        f"TOOL_BUDGET_REMAINING: {state.budget_remaining} (search costs {SEARCH_COST}, read costs {READ_COST}, "
        f"finish costs {FINISH_COST}; a document may be read only once and only after search returned it)",
        f"LLM_CALLS_REMAINING: {MAX_LLM_CALLS - state.llm_calls}",
        f"ACTION_HISTORY: {json.dumps(state.history)}",
        f"SEEN_DOCUMENTS (from search results): {json.dumps(seen)}",
    ]
    if llm_extracts:
        # The LLM sees RAW documents and its own memory; no Python-extracted evidence, no pre-resolved answer.
        parts.append("READ_DOCUMENTS (full text of every document you have read):\n"
                     + json.dumps([state.read_docs[i] for i in state.read_ids], indent=1))
        parts.append(f"MEMORY_FACTS (facts YOU recorded earlier that the controller verified as faithful): "
                     f"{json.dumps(state.llm_facts)}")
        if state.fact_feedback:
            parts.append("FACT_CHECK (entries of your last memory_update that were rejected):\n"
                         + "\n".join(state.fact_feedback))
        if state.last_observation is not None and "search_results" in state.last_observation:
            parts.append(f"LAST_OBSERVATION: {json.dumps(state.last_observation)}")
        elif state.last_observation is not None:
            parts.append("LAST_OBSERVATION: you just read the last document in READ_DOCUMENTS; "
                         "record its facts in memory_update.")
    else:  # ablation: Python extracts and resolves for the LLM (v1 design)
        parts.append(f"EVIDENCE (facts extracted from documents you have read): "
                     f"{json.dumps(state.evidence.annotated(state.query_date))}")
        if show_resolution:
            parts.append(f"ATTRIBUTE_STATUS (best current evidence per attribute, and seen-but-unread candidates): "
                         f"{json.dumps(attribute_status(state))}")
        if state.last_observation is not None:
            parts.append(f"LAST_OBSERVATION: {json.dumps(state.last_observation)}")
    if feedback:
        parts.append("CONTROLLER_FEEDBACK (your previous proposals in this step were rejected):\n" + "\n".join(feedback))
    parts.append(SCHEMA_X if llm_extracts else SCHEMA)
    return "\n\n".join(parts)
 
 
def build_rag_prompt(state: EpisodeState) -> str:
    docs = [state.read_docs[i] for i in state.read_ids]
    return "\n\n".join([
        "FINAL_ANSWER_ONLY. Answer the question using ONLY the documents below. No further retrieval is possible.",
        RULES, f"QUESTION: {state.question}", f"QUERY_DATE: {d2s(state.query_date)}",
        "DOCUMENTS:\n" + json.dumps(docs, indent=1),
        'Reply with ONLY a JSON object: {"action": "finish", "args": {"answer": <ANSWER>}, "justification": "..."}\n'
        'where <ANSWER> is "INSUFFICIENT EVIDENCE" or {"leader": "...", "department": "...", "budget_inr": <integer>, '
        '"approval_status": "APPROVED|PENDING|REJECTED", "support": {"leader": "<doc_id>", "department": "<doc_id>", '
        '"budget_inr": "<doc_id>", "approval_status": "<doc_id>"}}'])
 
 
def parse_llm_response(raw: str) -> Optional[dict]:
    """Strip <think> blocks / fences / chatter, return first JSON object that has an 'action' key."""
    if not raw:
        return None
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    dec = json.JSONDecoder()
    for m in re.finditer(r"\{", text):
        try:
            obj, _ = dec.raw_decode(text[m.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and "action" in obj:
            return obj
    return None
 
 
def normalize_answer(ans):
    """Lenient normalisation of an answer dict (for comparison only)."""
    if not isinstance(ans, dict):
        return None
    out = {}
    for a in ("leader", "department"):
        v = re.sub(r"\s+", " ", str(ans.get(a, "")).strip()).lower()
        if a == "department":
            v = re.sub(r"^department of\s+", "", v)
        out[a] = v
    b = ans.get("budget_inr")
    try:
        out["budget_inr"] = int(re.sub(r"[^\d]", "", str(b))) if not isinstance(b, bool) else None
    except ValueError:
        out["budget_inr"] = None
    out["approval_status"] = str(ans.get("approval_status", "")).strip().upper()
    return out
 
 
def verify_fact(fact, state: EpisodeState):
    """Faithfulness check of ONE LLM-recorded fact against the document text it cites.
    Checks only that the value is literally present in a document the agent has read. It does NOT judge
    whether the document is authoritative -- that is the LLM's reasoning and is scored post hoc."""
    if not isinstance(fact, dict):
        return False, "entry is not an object"
    attr, val, did = fact.get("attribute"), fact.get("value"), fact.get("doc_id")
    if attr not in ATTRS:
        return False, f"unknown attribute {attr!r}"
    if did not in state.read_docs:
        return False, f"{did!r} has not been read"
    text = state.read_docs[did]["content"]
    if attr in ("leader", "department"):
        ok = re.sub(r"\s+", " ", str(val).strip()).lower() in re.sub(r"\s+", " ", text).lower() and str(val).strip() != ""
    elif attr == "budget_inr":
        digits = re.sub(r"[^\d]", "", str(val)) if not isinstance(val, bool) else ""
        ok = digits != "" and int(digits) in {int(x.replace(",", "")) for x in _BUD.findall(text)}
    else:
        ok = str(val).strip().upper() in STATUSES and str(val).strip().upper() in text
    return (True, "") if ok else (False, f"value {val!r} for {attr} does not appear in {did}")
 
 
def record_memory_update(prop, state: EpisodeState):
    """Verify the optional memory_update of a valid proposal; store faithful facts, report the rest."""
    state.fact_feedback = []
    mu = prop.get("memory_update") if isinstance(prop, dict) else None
    if mu is None:
        return
    if not isinstance(mu, list):
        mu = [mu]
    for f in mu:
        ok, msg = verify_fact(f, state)
        if ok:
            doc = state.read_docs[f["doc_id"]]
            key = (f["attribute"], str(f["value"]), f["doc_id"])
            if key not in {(x["attribute"], str(x["value"]), x["doc_id"]) for x in state.llm_facts}:
                state.llm_facts.append({"attribute": f["attribute"], "value": f["value"], "doc_id": f["doc_id"],
                                        "doc_status": doc["status"], "effective": doc["effective_date"],
                                        "note": str(f.get("note", ""))[:120]})
        else:
            state.invalid.append({"step": None, "category": "unfaithful_fact", "message": msg, "proposal": f})
            state.fact_feedback.append(f"- rejected: {msg}")
 
 
def validate_proposal(prop, state: EpisodeState):
    """Deterministic gate. Returns (ok, category, message). Invalid proposals are NEVER executed."""
    if not isinstance(prop, dict):
        return False, "malformed", "Response was not a JSON object with an 'action' key."
    action, args = prop.get("action"), prop.get("args")
    if action not in ("search", "read", "finish"):
        return False, "unknown_action", f"Unknown action {action!r}. Use search, read or finish."
    if not isinstance(args, dict):
        return False, "schema", "'args' must be a JSON object."
    if action == "search":
        q, k = args.get("query"), args.get("k", TOP_K)
        if not isinstance(q, str) or not q.strip():
            return False, "schema", "search needs a non-empty string 'query'."
        if isinstance(k, bool) or not isinstance(k, int) or not 1 <= k <= TOP_K:
            return False, "schema", f"'k' must be an integer in 1..{TOP_K}."
        if SEARCH_COST > state.budget_remaining:
            return False, "over_budget", f"search costs {SEARCH_COST} but only {state.budget_remaining} remains."
    elif action == "read":
        did = args.get("doc_id")
        if not isinstance(did, str):
            return False, "schema", "read needs a string 'doc_id'."
        if did not in state.seen:
            return False, "unauthorized_read", f"{did!r} was never returned by a search; it cannot be read."
        if state.seen[did]["read"]:
            return False, "repeated_read", f"{did} was already read."
        if READ_COST > state.budget_remaining:
            return False, "over_budget", f"read costs {READ_COST} but only {state.budget_remaining} remains."
    else:  # finish
        ans = args.get("answer")
        if ans == ABSTAIN:
            return True, "ok", ""
        if not isinstance(ans, dict):
            return False, "schema", f"'answer' must be {ABSTAIN!r} or an answer object."
        missing = [a for a in ATTRS if a not in ans]
        sup = ans.get("support")
        if missing or not isinstance(sup, dict) or any(a not in sup for a in ATTRS):
            return False, "schema", f"answer needs keys {ATTRS} and a 'support' object with the same keys."
        bad = [v for v in sup.values() if v not in state.read_ids]
        if bad:
            return False, "unsupported_citation", f"support cites documents you have not read: {bad}."
    return True, "ok", ""

In [ ]:
def answer_from_resolution(state: EpisodeState):
    best = state.evidence.resolve(state.query_date)
    if any(best[a] is None for a in ATTRS):
        return ABSTAIN
    ans = {a: best[a].value for a in ATTRS}
    ans["support"] = {a: best[a].doc_id for a in ATTRS}
    return ans


def _act(action, args, why):
    prop = {"action": action, "args": args, "justification": why}
    return json.dumps(prop), prop


class LLMPolicy:
    """pi_LLM(M_t, B_t): prompt -> client -> JSON. The LLM only PROPOSES; the controller validates."""
    name, uses_llm, retry_on_invalid, llm_cap = "llm", True, True, MAX_LLM_CALLS

    def __init__(self, client, show_resolution=True, save_prompts=False, llm_extracts=True):
        self.client, self.show_resolution, self.save_prompts = client, show_resolution, save_prompts
        self.llm_extracts = llm_extracts
        self.last_prompt = None

    def can_propose(self, state):
        return state.llm_calls < self.llm_cap

    def propose(self, state, feedback):
        prompt = build_agent_prompt(state, feedback, self.show_resolution, self.llm_extracts)
        self.last_prompt = prompt
        state.llm_calls += 1
        if hasattr(self.client, "bind"):
            self.client.bind(state)
        raw = self.client.complete(prompt, temperature=0.0 if not feedback else 0.2)
        return raw, parse_llm_response(raw)


class RAGPolicy:
    """Baseline 1: one search with the original question, read top-3, ONE LLM call to answer."""
    name, uses_llm, retry_on_invalid, llm_cap = "rag", True, False, 1

    def __init__(self, client, save_prompts=False):
        self.client, self.save_prompts, self.last_prompt = client, save_prompts, None

    def can_propose(self, state):
        return state.n_search == 0 or state.n_read < min(3, len(state.search_log[0]["ids"])) or state.llm_calls < 1

    def propose(self, state, feedback):
        if state.n_search == 0:
            return _act("search", {"query": state.question, "k": TOP_K}, "fixed RAG: search the question")
        top = state.search_log[0]["ids"][:3]
        nxt = [i for i in top if not state.seen[i]["read"]]
        if nxt:
            return _act("read", {"doc_id": nxt[0]}, "fixed RAG: read top-3")
        prompt = build_rag_prompt(state)
        self.last_prompt = prompt
        state.llm_calls += 1
        if hasattr(self.client, "bind"):
            self.client.bind(state)
        raw = self.client.complete(prompt, temperature=0.0)
        return raw, parse_llm_response(raw)


class RulePolicy:
    """Baseline 2: hand-written rules over project id, document type and effective dates. No LLM."""
    name, uses_llm, retry_on_invalid, llm_cap = "rule", False, False, 0
    QUERIES = {"REG": "{pid} project registration leader department",
               "BUD": "{pid} budget authorization", "APP": "{pid} approval status notice"}

    def can_propose(self, state):
        return True

    def propose(self, state, feedback=None):
        pid, d = state.pid, state.query_date
        res = state.evidence.resolve(d)
        rev_first = d >= REVISION_DATE
        plan = [("leader", "REG", ["REG"]), ("department", "REG", ["REG"]),
                ("budget_inr", "BUD", ["BUD_REV", "BUD_INIT"] if rev_first else ["BUD_INIT", "BUD_REV"]),
                ("approval_status", "APP", ["APP_REV", "APP_INIT"] if rev_first else ["APP_INIT", "APP_REV"])]
        done_queries = {s["query"] for s in state.search_log}
        read_types = {classify_title(state.seen[i]["title"]) for i in state.read_ids}
        for attr, group, order in plan:
            if res[attr] is not None:
                continue
            for t in order:
                if t in read_types:
                    continue
                cand = [i for i, m in state.seen.items() if classify_title(m["title"]) == t
                        and not m["read"] and m["status"] == "ISSUED" and f"({pid})" in m["title"]]
                if cand and state.budget_remaining >= READ_COST:
                    return _act("read", {"doc_id": cand[0]}, f"rule: read {t} for {attr}")
                q = self.QUERIES[group].format(pid=pid)
                if not cand and q not in done_queries and state.budget_remaining >= SEARCH_COST:
                    return _act("search", {"query": q, "k": TOP_K}, f"rule: locate {group} documents")
        return _act("finish", {"answer": answer_from_resolution(state)}, "rule: evidence complete or exhausted")

class OpenRouterClient:
    """Zero-dependency OpenRouter API client targeting lightweight Qwen models (e.g. Qwen3 A3B Instruct / Qwen 4B).

    Reads OPENROUTER_API_KEY from environment or /Users/kartikgoyal/Desktop/BTP/.env.
    """

    def __init__(self, api_key: Optional[str] = None, model: Optional[str] = None, pace: float = 1.5):
        self.base_url = "https://openrouter.ai/api/v1"
        self.api_key = api_key or os.environ.get("OPENROUTER_API_KEY") or os.environ.get("LLM_API_KEY")
        if not self.api_key:
            for ep in ["/Users/kartikgoyal/Desktop/BTP/.env", os.path.expanduser("~/Desktop/BTP/.env"), ".env", "../.env", "../../.env"]:
                if os.path.exists(ep):
                    try:
                        with open(ep) as f:
                            for line in f:
                                if line.strip().startswith("OPENROUTER_API_KEY="):
                                    k = line.strip().split("=", 1)[1].strip().strip("'").strip('"')
                                    if k:
                                        self.api_key = k
                                        os.environ["OPENROUTER_API_KEY"] = k
                                        break
                    except Exception:
                        pass
                if self.api_key:
                    break
        self.model = model or os.environ.get("OPENROUTER_MODEL") or os.environ.get("LLM_MODEL") or "qwen/qwen3-30b-a3b-instruct-2507"
        self.pace = pace  # 1.5s delay prevents 429 rate limit bursts on OpenRouter

        if not self.api_key:
            raise ValueError(
                "OPENROUTER_API_KEY is not set.\nPlease fill it in /Users/kartikgoyal/Desktop/BTP/.env or run: export OPENROUTER_API_KEY='sk-or-v1-...'"
            )

    def complete(self, prompt: str, temperature: float = 0.0, max_tokens: int = 1024) -> str:
        payload = json.dumps({
            "model": self.model,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}],
            "response_format": {"type": "json_object"}
        }).encode("utf-8")

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://github.com/karlw2805/BTP",
            "X-Title": "IITR BTP Agentic Systems",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        }

        for attempt in range(8):
            try:
                time.sleep(self.pace)
                req = urllib.request.Request(
                    f"{self.base_url}/chat/completions",
                    data=payload,
                    headers=headers
                )
                with urllib.request.urlopen(req, timeout=60) as resp:
                    data = json.loads(resp.read().decode("utf-8"))
                    return data["choices"][0]["message"]["content"]
            except urllib.error.HTTPError as e:
                err_body = ""
                try:
                    err_body = e.read().decode("utf-8")
                except Exception:
                    pass
                if e.code == 429 and attempt < 7:
                    wait_time = 5.0 * (attempt + 1)
                    header_retry = e.headers.get("Retry-After") or e.headers.get("retry-after") if hasattr(e, "headers") else None
                    if header_retry:
                        try:
                            wait_time = max(wait_time, float(header_retry) + 1.0)
                        except ValueError:
                            pass
                    m = re.search(r"try again in (?:(\d+)m)?([\d\.]+)s", err_body)
                    if m:
                        try:
                            mins = int(m.group(1)) if m.group(1) else 0
                            secs = float(m.group(2))
                            wait_time = max(wait_time, mins * 60 + secs + 2.0)
                        except ValueError:
                            pass
                    print(f"    [OpenRouter Rate-Limit Auto-Throttle] HTTP 429: Backing off for {wait_time:.1f}s...")
                    time.sleep(wait_time)
                    continue
                if e.code in (500, 502, 503) and attempt < 7:
                    wait_time = 4.0 * (attempt + 1)
                    print(f"    [OpenRouter Server Notice] HTTP {e.code} ({e.reason}). Retrying in {wait_time:.1f}s...")
                    time.sleep(wait_time)
                    continue
                raise RuntimeError(f"OpenRouter API Error {e.code}: {e.reason} - {err_body}") from e
            except (TimeoutError, urllib.error.URLError, OSError) as e:
                if attempt < 7:
                    wait_time = 3.0 * (attempt + 1)
                    print(f"    [Network Notice] {e}. Retrying in {wait_time:.1f}s...")
                    time.sleep(wait_time)
                    continue
                raise


class GroqClient(OpenRouterClient):
    """Fallback client for Groq targeting qwen/qwen3.8-27b if desired."""
    def __init__(self, api_key: Optional[str] = None, model: Optional[str] = None, pace: float = 2.0):
        self.base_url = "https://api.groq.com/openai/v1"
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        self.model = model or "qwen/qwen3.8-27b"
        self.pace = pace


def get_client(model: Optional[str] = None):
    """Instantiate the active API client, preferring OpenRouter over Groq."""
    load_btp_env()
    or_key = os.environ.get("OPENROUTER_API_KEY")
    groq_key = os.environ.get("GROQ_API_KEY")
    if or_key:
        return OpenRouterClient(api_key=or_key, model=model)
    elif groq_key:
        return GroqClient(api_key=groq_key, model=model)
    else:
        return OpenRouterClient(model=model)


class CachedClient:
    """Disk cache keyed by (model, temperature, prompt) -> reruns are free, instant, and reproducible."""

    def __init__(self, inner, path="llm_cache.json"):
        self.inner, self.path = inner, path
        self.cache = json.load(open(path)) if os.path.exists(path) else {}

    def bind(self, state):
        if hasattr(self.inner, "bind"):
            self.inner.bind(state)

    def complete(self, prompt, temperature=0.0, max_tokens=1024):
        key = hashlib.sha256(f"{getattr(self.inner, 'model', '')}|{temperature}|{prompt}".encode()).hexdigest()
        if key not in self.cache:
            self.cache[key] = self.inner.complete(prompt, temperature=temperature, max_tokens=max_tokens)
            json.dump(self.cache, open(self.path, "w"), indent=2)
        return self.cache[key]


In [11]:
def run_episode(policy, docs, retriever, hidden, pid, d, budget=TOOL_BUDGET, keep_prompts=False):
    name = hidden.projects[pid]["name"]
    state = EpisodeState(make_question(pid, name, d), pid, d, budget)
    env = ToolEnv(docs, retriever)
    trace, feedback, answer, termination = [], [], ABSTAIN, "finish"
    for step in range(1, 80):
        if not policy.can_propose(state):
            termination = "llm_call_limit"
            break
        raw, prop = policy.propose(state, feedback)
        ok, cat, msg = validate_proposal(prop, state)
        entry = {"step": step, "llm_calls": state.llm_calls, "raw": raw, "proposal": prop,
                 "valid": ok, "category": cat, "budget_before": state.budget_remaining}
        if keep_prompts and getattr(policy, "last_prompt", None):
            entry["prompt"] = policy.last_prompt
        if not ok:
            state.invalid.append({"step": step, "category": cat, "message": msg, "proposal": prop})
            entry["message"] = msg
            trace.append(entry)
            if policy.retry_on_invalid:
                feedback.append(f"- attempt {len(feedback) + 1}: rejected ({cat}): {msg}")
                continue
            termination = "scripted_invalid"
            break
        feedback = []
        if getattr(policy, "llm_extracts", False):
            record_memory_update(prop, state)
        act, args = prop["action"], prop["args"]
        if act == "finish":
            answer = args["answer"]
            trace.append(entry)
            break
        if act == "search":
            obs = env.search(args["query"], args.get("k", TOP_K))
            state.budget_remaining -= SEARCH_COST
            state.total_cost += SEARCH_COST
            state.n_search += 1
            state.search_log.append({"query": args["query"], "ids": [o["doc_id"] for o in obs]})
            for o in obs:
                state.seen.setdefault(o["doc_id"], {**{k: o[k] for k in ("title", "issue_date", "status")},
                                                    "read": False})
            state.history.append({"action": "search", "query": args["query"], "returned": [o["doc_id"] for o in obs]})
            state.last_observation = {"search_results": obs}
        else:
            doc = env.read(args["doc_id"])
            state.budget_remaining -= READ_COST
            state.total_cost += READ_COST
            state.n_read += 1
            state.seen[doc["doc_id"]]["read"] = True
            state.read_ids.append(doc["doc_id"])
            state.read_docs[doc["doc_id"]] = doc
            state.evidence.add_document(doc)
            state.history.append({"action": "read", "doc_id": doc["doc_id"]})
            state.last_observation = {"document": doc}
        entry["observation"] = state.last_observation
        entry["budget_after"] = state.budget_remaining
        trace.append(entry)
    else:
        termination = "step_limit"
    return finalize(state, env, answer, termination, trace, hidden, docs, policy)

In [12]:
FAILURE_ORIGINS = {
    "reliance_on_draft": (
        "LLM Policy / Reasoning Failure: The model accepted claims from a DRAFT memorandum, "
        "violating Rule 1 (only ISSUED documents are authoritative)."
    ),
    "failure_to_recognize_revision": (
        "LLM Policy / Epistemic Failure: The query date was on or after May 1, 2026, but the model "
        "retained outdated initial authorization values instead of searching/reading the revision."
    ),
    "used_not_yet_effective_revision": (
        "LLM Policy / Temporal Reasoning Failure: The model retrieved a revised document whose "
        "effective date (May 1, 2026) is after the query date (March 1, 2026), violating Rule 2."
    ),
    "retrieval_failure": (
        "Retrieval Limitation: TF-IDF retrieval ranked irrelevant or competing documents above the "
        "required authoritative source, so the agent never observed the document in search results."
    ),
    "incorrect_document_selection": (
        "LLM Policy / Action Selection Failure: The agent observed the relevant document among search "
        "results but chose to read unrelated or draft documents instead, exhausting opportunities."
    ),
    "budget_exhaustion": (
        "Resource Constraint / Budget Exhaustion: The agent expended its tool budget (B=12) through "
        "redundant searches or non-pertinent reads before obtaining all required sources."
    ),
    "llm_call_exhaustion": (
        "Resource Constraint / Control Logic: The agent reached the maximum limit of 15 LLM calls "
        "without producing a terminating finish action."
    ),
    "hallucinated_fact": (
        "LLM Policy / Hallucination: The agent asserted an attribute value that did not appear in any "
        "document it actually read."
    ),
    "ungrounded_citation": (
        "LLM Policy / Grounding Verification Failure: The agent submitted an answer whose supporting "
        "citations cited unread, draft, or superseded documents."
    ),
    "insufficient_evidence": (
        "LLM Policy / Premature Abstention: The agent chose to abstain (INSUFFICIENT EVIDENCE) even though "
        "candidate documents had been retrieved by search and budget was still available."
    ),
    "premature_stopping": (
        "LLM Policy / Premature Stopping: The agent finished with missing or incomplete attributes while "
        "sufficient tool budget (>=2) remained to read candidate documents."
    ),
    "malformed_final_answer": (
        "LLM Policy / Schema Non-compliance: The final answer could not be parsed into the required JSON schema."
    ),
}


def verify_answer(answer, pid, d, hidden, docs, read_ids):
    """Returns (exact, grounded, reasons). Uses hidden records; called only after the agent has stopped."""
    truth = hidden.truth(pid, d)
    norm = normalize_answer(answer)
    if norm is None:
        return False, False, ["abstained_or_malformed"]
    exact = (norm["leader"] == truth["leader"].lower() and norm["department"] == truth["department"].lower()
             and norm["budget_inr"] == truth["budget_inr"] and norm["approval_status"] == truth["approval_status"])
    reasons, grounded = [], True
    sup = answer.get("support", {})
    claim = {"leader": norm["leader"], "department": norm["department"], "budget_inr": norm["budget_inr"],
             "approval_status": norm["approval_status"]}
    for a in ATTRS:
        did = sup.get(a)
        if did not in read_ids:
            grounded = False; reasons.append(f"{a}: cited doc {did} not actually read"); continue
        doc, (dpid, dtype) = docs[did], hidden.doc_meta[did]
        vals = hidden.doc_values[did]
        if doc.status != "ISSUED":
            grounded = False; reasons.append(f"{a}: {did} is a DRAFT"); continue
        if dpid != pid or a not in vals:
            grounded = False; reasons.append(f"{a}: {did} is not a {a} source for {pid}"); continue
        applicable_from = s2d(doc.effective_date or doc.issue_date)
        if applicable_from > d:
            grounded = False; reasons.append(f"{a}: {did} not yet effective on {d}"); continue
        later = [o for o, (p, t) in hidden.doc_meta.items()
                 if p == pid and o != did and docs[o].status == "ISSUED" and a in hidden.doc_values[o]
                 and d >= s2d(docs[o].effective_date or docs[o].issue_date) > applicable_from]
        if later:
            grounded = False; reasons.append(f"{a}: {did} superseded by {later[0]}"); continue
        v = vals[a]
        v = v.lower() if isinstance(v, str) and a in ("leader", "department") else v
        if a == "department" and isinstance(v, str):
            v = re.sub(r"^department of\s+", "", v)
        claim_v = claim[a]
        if a == "department" and isinstance(claim_v, str):
            claim_v = re.sub(r"^department of\s+", "", claim_v)
        if claim_v != v:
            grounded = False; reasons.append(f"{a}: claim does not match content of {did}")
    return exact, grounded, reasons


def classify_failure(r, hidden):
    """Heuristic first-pass labels (to be refined by manual trace reading)."""
    if r["success"]:
        return None
    pid, d = r["pid"], s2d(r["query_date"])
    need = set(hidden.source_docs(pid, d).values())
    seen, read = set(r["seen_ids"]), set(r["read_ids"])
    ans = r["answer"]
    if r["abstained"]:
        if r["termination"] == "llm_call_limit":
            return "llm_call_exhaustion"
        if r["budget_exhausted"]:
            return "budget_exhaustion"
        if need - seen:
            return "retrieval_failure"
        return "insufficient_evidence"
    norm, truth, memo = normalize_answer(ans), hidden.truth(pid, d), hidden.memo_values(pid)
    if norm is None:
        return "malformed_final_answer"
    if r["exact"] and not r["grounded"]:
        return "ungrounded_citation"
    p = hidden.projects[pid]
    for a in ATTRS:
        mv = memo[a].lower() if isinstance(memo[a], str) and a in ("leader", "department") else memo[a]
        if norm[a] == mv:
            return "reliance_on_draft"
    if d >= REVISION_DATE and (norm["budget_inr"] == p["budget_init"] or norm["approval_status"] == p["approval_init"]):
        return "failure_to_recognize_revision"
    if d < REVISION_DATE and (norm["budget_inr"] == p["budget_rev"] or norm["approval_status"] == p["approval_rev"]):
        return "used_not_yet_effective_revision"
    read_vals = [hidden.doc_values[i] for i in read]
    for a in ATTRS:
        v = norm[a]
        if not any((x[a].lower() if isinstance(x.get(a), str) and a in ("leader", "department") else x.get(a)) == v
                   for x in read_vals if a in x):
            return "hallucinated_fact"
    if need - read:
        return "retrieval_failure" if need - seen else ("premature_stopping" if r["budget_remaining"] >= 2
                                                        else "budget_exhaustion")
    return "incorrect_document_selection"


def inject_failure_analysis(res, hidden):
    """Inject diagnostic root-cause and discrepancy analysis into an episode result."""
    if res["success"]:
        return
    cat = res.get("failure_category")
    truth = hidden.truth(res["pid"], s2d(res["query_date"]))
    norm = normalize_answer(res["answer"]) or {}
    discrepancies = {}
    for a in ATTRS:
        tv = truth.get(a)
        tv_comp = tv.lower() if isinstance(tv, str) and a in ("leader", "department") else tv
        cv = norm.get(a)
        if cv != tv_comp:
            discrepancies[a] = {"claimed": norm.get(a), "true": tv}

    res["failure_analysis"] = {
        "failure_category": cat,
        "origin_explanation": FAILURE_ORIGINS.get(cat, "Unclassified failure mode."),
        "grounding_violations": res.get("grounding_reasons", []),
        "attribute_discrepancies": discrepancies,
    }


def finalize(state, env, answer, termination, trace, hidden, docs, policy):
    abstained = answer == ABSTAIN or normalize_answer(answer) is None
    exact, grounded, reasons = verify_answer(answer, state.pid, state.query_date, hidden, docs, env.read_ids)
    success = exact and grounded
    cats = Counter(i["category"] for i in state.invalid)
    n_inv = len(state.invalid)
    res = {
        "method": policy.name, "pid": state.pid, "query_date": d2s(state.query_date),
        "question": state.question,
        "answer": answer, "truth": hidden.truth(state.pid, state.query_date),
        "abstained": abstained, "exact": exact, "grounded": grounded, "success": success,
        "grounding_reasons": reasons, "termination": termination,
        "total_cost": state.total_cost, "n_search": state.n_search, "n_read": state.n_read,
        "budget_remaining": state.budget_remaining,
        "budget_exhausted": state.budget_remaining <= 0 or cats.get("over_budget", 0) > 0,
        "n_llm_calls": state.llm_calls, "n_invalid": n_inv, "invalid_by_category": dict(cats),
        "n_malformed": cats.get("malformed", 0) + cats.get("schema", 0) + cats.get("unknown_action", 0),
        "n_unfaithful": cats.get("unfaithful_fact", 0), "llm_facts": state.llm_facts,
        "n_unauthorized": cats.get("unauthorized_read", 0) + cats.get("repeated_read", 0)
        + cats.get("unsupported_citation", 0),
        "seen_ids": list(state.seen), "read_ids": list(env.read_ids),
        "J": FAIL_PENALTY * (0 if success else 1) + 0.1 * state.total_cost + 0.02 * state.llm_calls,
        "trace": trace,
    }
    res["failure_category"] = classify_failure(res, hidden)
    inject_failure_analysis(res, hidden)
    return res


def summarize(results):
    n = len(results)
    avg = lambda k: sum(r[k] for r in results) / n
    return {"method": results[0]["method"], "n": n,
            "exact_acc": avg("exact"), "grounded_acc": avg("grounded"), "success": avg("success"),
            "abstain_rate": avg("abstained"), "avg_cost": avg("total_cost"), "avg_search": avg("n_search"),
            "avg_read": avg("n_read"), "budget_exhausted": avg("budget_exhausted"),
            "avg_llm_calls": avg("n_llm_calls"), "malformed": sum(r["n_malformed"] for r in results),
            "unauthorized": sum(r["n_unauthorized"] for r in results),
            "unfaithful": sum(r["n_unfaithful"] for r in results),
            "invalid_total": sum(r["n_invalid"] for r in results), "avg_J": avg("J")}


def print_table(summaries):
    cols = [("method", "{:<8}"), ("exact_acc", "{:>7.3f}"), ("grounded_acc", "{:>7.3f}"), ("success", "{:>7.3f}"),
            ("abstain_rate", "{:>7.3f}"), ("avg_cost", "{:>7.2f}"), ("avg_search", "{:>7.2f}"),
            ("avg_read", "{:>7.2f}"), ("budget_exhausted", "{:>7.3f}"), ("avg_llm_calls", "{:>7.2f}"),
            ("malformed", "{:>7d}"), ("unauthorized", "{:>7d}"), ("unfaithful", "{:>7d}"), ("avg_J", "{:>7.2f}")]
    heads = ["method", "exact", "ground", "success", "abstain", "cost", "search", "read", "budgetX", "llm",
             "malf", "unauth", "unfaith", "J"]
    print(" ".join(f"{h:>7}" if i else f"{h:<8}" for i, h in enumerate(heads)))
    for s in summaries:
        print(" ".join(f.format(s[k]) for k, f in cols))


In [ ]:
def all_questions():
    return [(f"P{i + 1:02d}", d) for i in range(N_PROJECTS) for d in QUERY_DATES]


def evaluate(policy, docs, retriever, hidden, keep_prompts=False):
    return [run_episode(policy, docs, retriever, hidden, pid, d, keep_prompts=keep_prompts)
            for pid, d in all_questions()]


def save_run_logs(results, base_dir: str = "runs", seed: Optional[int] = None):
    """Save both individual per-question JSON files and consolidated JSONL trace."""
    s = SEED if seed is None else seed
    method = results[0]["method"]
    run_dir = os.path.join(base_dir, f"Run_Seed_{s}")
    method_dir = os.path.join(run_dir, method)
    os.makedirs(method_dir, exist_ok=True)

    # 1. Save individual per-question JSON files with full trace & failure analysis
    for r in results:
        fname = f"{r['pid']}_{r['query_date']}.json"
        fpath = os.path.join(method_dir, fname)
        with open(fpath, "w") as f:
            json.dump(r, f, indent=2)

    # 2. Save consolidated JSONL trace for bulk analysis & reproducibility
    trace_path = os.path.join(run_dir, f"{method}_traces.jsonl")
    with open(trace_path, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")

def smoke_test(seed: Optional[int] = None, method: str = "llm", model: Optional[str] = None):
    """Live network test: runs 1 episode directly against OpenRouter (or Groq) without caching or file logging."""
    s = SEED if seed is None else seed
    load_btp_env()
    api_key = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("GROQ_API_KEY") or os.environ.get("LLM_API_KEY")
    if not api_key:
        print("\n" + "!" * 70)
        print("ERROR: OPENROUTER_API_KEY is not set.")
        print("Please set your OpenRouter API key in /Users/kartikgoyal/Desktop/BTP/.env:")
        print("    OPENROUTER_API_KEY='sk-or-v1-...'")
        print("!" * 70 + "\n")
        return None

    # Use raw client directly (NO CachedClient) -> 100% live API test, no cache pollution
    live_client = get_client(model)
    docs, hidden = build_corpus(s)
    ret = Retriever(docs)

    policy = LLMPolicy(live_client) if method == "llm" else RAGPolicy(live_client)
    target_date = QUERY_DATES[1]  # June 1, 2026 tests revision handling

    print("=" * 70)
    print("LIVE SMOKE TEST (Uncached, No Log Files Created)")
    print(f"Target Model : {live_client.model} via {live_client.base_url}")
    print(f"Test Episode : {method.upper()} on Project P01 @ {d2s(target_date)}")
    print("=" * 70)

    res = run_episode(policy, docs, ret, hidden, "P01", target_date, keep_prompts=True)

    print(f"Live API Status : CONNECTED & RESPONDING")
    print(f"Result          : Success={res['success']} | Exact={res['exact']} | Grounded={res['grounded']}")
    print(f"Tool Cost       : {res['total_cost']}/{TOOL_BUDGET} | LLM Calls: {res['n_llm_calls']}/{MAX_LLM_CALLS}")
    print(f"Invalid Actions : {res['n_invalid']} (category: {res['invalid_by_category']})")
    print(f"Submitted Answer: {res['answer']}")
    if not res["success"]:
        print(f"Grounding Note  : {res['grounding_reasons']}")
        print(f"Failure Class   : {res['failure_category']}")
        origin = FAILURE_ORIGINS.get(res["failure_category"], "Unclassified")
        print(f"Diagnosis       : {origin}")
    print("=" * 70)
    print("Smoke test complete. Zero files logged, zero cache entries written.")
    print("=" * 70)
    return res

def selftest(seed: Optional[int] = None):
    s = SEED if seed is None else seed
    docs, hidden = build_corpus(s)
    ret = Retriever(docs)
    assert len(docs) == 72 and len({d.doc_id for d in docs.values()}) == 72
    for pid in hidden.projects:
        types = sorted(t for (p, t) in hidden.doc_meta.values() if p == pid)
        assert types == sorted(["REG", "BUD_INIT", "BUD_REV", "APP_INIT", "APP_REV", "MEMO"]), pid
    assert all(d.status in ("ISSUED", "DRAFT") for d in docs.values())
    assert sum(d.status == "DRAFT" for d in docs.values()) == 12
    for pid in hidden.projects:      # answer genuinely depends on the date
        t1, t2 = hidden.truth(pid, QUERY_DATES[0]), hidden.truth(pid, QUERY_DATES[1])
        assert t1["budget_inr"] != t2["budget_inr"] and t1["approval_status"] != t2["approval_status"]
    print("ok corpus: 72 docs, 6/project, date-dependent truth")

    env = ToolEnv(docs, ret)
    hits = env.search("P03 budget authorization", 5)
    assert len(hits) <= 5 and set(hits[0]) == {"doc_id", "title", "issue_date", "status"}
    try:
        env.read("DOC-001" if "DOC-001" not in env.returned_ids else "DOC-072"); assert False
    except PermissionError:
        pass
    env.read(hits[0]["doc_id"])
    try:
        env.read(hits[0]["doc_id"]); assert False
    except PermissionError:
        pass
    print("ok tools: k<=5, read-after-search enforced, repeat read blocked")

    st = EpisodeState("q", "P01", QUERY_DATES[0])
    cases = [(None, "malformed"), ({"action": "dance", "args": {}}, "unknown_action"),
             ({"action": "search", "args": {"query": ""}}, "schema"),
             ({"action": "read", "args": {"doc_id": "DOC-001"}}, "unauthorized_read"),
             ({"action": "finish", "args": {"answer": {"leader": "x"}}}, "schema")]
    for prop, cat in cases:
        assert validate_proposal(prop, st)[1] == cat, (prop, cat)
    st.seen["DOC-001"] = {"title": "x", "issue_date": "", "status": "ISSUED", "read": False}
    st.budget_remaining = 1
    assert validate_proposal({"action": "read", "args": {"doc_id": "DOC-001"}}, st)[1] == "over_budget"
    st.budget_remaining = 12
    st.seen["DOC-001"]["read"] = True
    assert validate_proposal({"action": "read", "args": {"doc_id": "DOC-001"}}, st)[1] == "repeated_read"
    print("ok controller validation categories")

    res = evaluate(RulePolicy(), docs, ret, hidden)
    s = summarize(res)
    assert s["success"] == 1.0 and s["avg_cost"] <= TOOL_BUDGET, s
    print(f"ok rule agent: success={s['success']:.2f} avg_cost={s['avg_cost']:.2f}")

    # LLM-extraction path test: faithful facts stored, unfaithful ones rejected
    st = EpisodeState("q", "P01", QUERY_DATES[1])
    did = next(i for i, (pp, tt) in hidden.doc_meta.items() if pp == "P01" and tt == "BUD_REV")
    st.read_docs[did] = docs[did].public(); st.read_ids.append(did)
    good = hidden.doc_values[did]["budget_inr"]
    record_memory_update({"memory_update": [
        {"attribute": "budget_inr", "value": good, "doc_id": did, "note": "ok"},
        {"attribute": "budget_inr", "value": good + 1, "doc_id": did, "note": "wrong"},
        {"attribute": "leader", "value": "Nobody", "doc_id": did},
        {"attribute": "budget_inr", "value": good, "doc_id": "DOC-NOPE"}]}, st)
    assert len(st.llm_facts) == 1 and len(st.invalid) == 3 and len(st.fact_feedback) == 3
    pr = build_agent_prompt(st, [], True, True)
    assert "READ_DOCUMENTS" in pr and "ATTRIBUTE_STATUS" not in pr and '"standing_on_query_date"' not in pr

    # Leakage check: secrets from other projects must not be visible
    p5 = hidden.projects["P05"]
    secrets = [p5["name"], p5["leader"], str(p5["budget_init"]), str(p5["budget_rev"])]
    assert not any(x in pr for x in secrets), "prompt leaks another project's secrets"
    assert "doc_ids" not in pr
    print("ok prompt leakage test: no hidden records leaked")
    print("ALL OFFLINE SELF-TESTS PASSED")

def main(seed: Optional[int] = None,
         methods: Optional[list] = None,
         model: Optional[str] = None,
         out: str = "runs",
         selftest_flag: bool = False,
         smoke_test_flag: bool = False,
         python_extract: bool = False,
         no_resolution: bool = False,
         args: Optional[list] = None):
    """Unified entry point: accepts direct programmatic kwargs or CLI flags, defaulting to global SEED."""
    is_jupyter = "ipykernel" in sys.modules
    if not is_jupyter or args is not None:
        ap = argparse.ArgumentParser(description="Task 3: Evidence-grounded Autonomous Research Agent")
        ap.add_argument("--selftest", action="store_true", help="run offline self-tests and exit")
        ap.add_argument("--smoke-test", action="store_true", help="run a live, uncached smoke test and exit")
        ap.add_argument("--model", default=None, help="LLM model slug (default: qwen/qwen3-30b-a3b-instruct-2507)")
        ap.add_argument("--methods", nargs="+", default=["rule", "rag", "llm"],
                        choices=["rule", "rag", "llm"], help="methods to evaluate")
        ap.add_argument("--seed", type=int, default=SEED, help=f"corpus random seed (default: {SEED})")
        ap.add_argument("--out", default="runs", help="base directory for runs and logs")
        ap.add_argument("--python-extract", action="store_true", help="ablation: Python extracts+resolves (v1)")
        ap.add_argument("--no-resolution", action="store_true", help="ablation: hide ATTRIBUTE_STATUS from prompt")
        cli = ap.parse_args(args)

        active_seed = seed if seed is not None else cli.seed
        active_methods = methods if methods is not None else cli.methods
        active_model = model if model is not None else cli.model
        active_out = out if out != "runs" else cli.out
        active_selftest = selftest_flag or cli.selftest
        active_smoke_test = smoke_test_flag or cli.smoke_test
        active_python_extract = python_extract or cli.python_extract
        active_no_resolution = no_resolution or cli.no_resolution
    else:
        active_seed = SEED if seed is None else seed
        active_methods = ["rule", "rag", "llm"] if methods is None else methods
        active_model = model
        active_out = out
        active_selftest = selftest_flag
        active_smoke_test = smoke_test_flag
        active_python_extract = python_extract
        active_no_resolution = no_resolution

    if active_selftest:
        return selftest(active_seed)

    if active_smoke_test:
        return smoke_test(active_seed, model=active_model)

    docs, hidden = build_corpus(active_seed)
    ret = Retriever(docs)

    client = None
    if any(m in ("rag", "llm") for m in active_methods):
        load_btp_env()
        api_key = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("GROQ_API_KEY") or os.environ.get("LLM_API_KEY")
        if not api_key:
            print("\n" + "!" * 70)
            print("ERROR: OPENROUTER_API_KEY is not set.")
            print("Please set your OpenRouter API key in /Users/kartikgoyal/Desktop/BTP/.env:")
            print("!" * 70 + "\n")
            return
        client = CachedClient(get_client(active_model))

    policies = {
        "rule": RulePolicy,
        "rag": lambda: RAGPolicy(client),
        "llm": lambda: LLMPolicy(client, show_resolution=not active_no_resolution,
                                 llm_extracts=not active_python_extract)
    }

    # Execute full benchmark across all 24 questions
    run_dir = os.path.join(active_out, f"Run_Seed_{active_seed}")
    os.makedirs(run_dir, exist_ok=True)
    summaries, failures = [], []

    for m in active_methods:
        print(f"Evaluating {m.upper()} on Seed {active_seed} (24 questions)...")
        res = evaluate(policies[m](), docs, ret, hidden, keep_prompts=(m == "llm"))
        save_run_logs(res, active_out, active_seed)
        summaries.append(summarize(res))
        for r in res:
            if not r["success"]:
                failures.append(r)

    # Print consolidated benchmark table
    keys = ["method", "exact_acc", "grounded_acc", "success", "abstain_rate",
            "avg_cost", "avg_search", "avg_read", "budget_exhausted",
            "avg_llm_calls", "malformed", "unauthorized", "unfaithful", "avg_J"]
    headers = ["method", "exact", "ground", "success", "abstain", "cost",
               "search", "read", "budgetX", "llm", "malf", "unauth", "unfaith", "J"]

    print("\n" + "=" * 70)
    print(f"BENCHMARK SUMMARY TABLE (Seed: {active_seed}, 24 questions)")
    print("=" * 70)
    print("{:<10} {:>6} {:>6} {:>7} {:>7} {:>7} {:>7} {:>7} {:>7} {:>7} {:>6} {:>7} {:>7} {:>7}".format(*headers))
    for s in summaries:
        row = [s["method"]]
        for k in keys[1:]:
            v = s.get(k, 0)
            if isinstance(v, float):
                row.append(f"{v:6.3f}" if k not in ("avg_cost", "avg_search", "avg_read", "avg_llm_calls", "avg_J") else f"{v:6.2f}")
            else:
                row.append(f"{v:>6}")
        print("{:<10} {:>6} {:>6} {:>7} {:>7} {:>7} {:>7} {:>7} {:>7} {:>7} {:>6} {:>7} {:>7} {:>7}".format(*row))

    with open(os.path.join(run_dir, "summary.json"), "w") as f:
        json.dump(summaries, f, indent=2)

    # Print failure diagnosis breakdown
    if failures:
        print("\n" + "=" * 70)
        print("FAILURE CATEGORIES BREAKDOWN")
        print("=" * 70)
        cat_counts = Counter(r["failure_category"] for r in failures)
        for cat, cnt in cat_counts.most_common():
            print(f"  - {cat:<30}: {cnt} episodes")

        print("\n" + "=" * 70)
        print("FAILED EPISODES LOG REGISTER (Detailed analysis injected into each file)")
        print("=" * 70)
        for r in failures:
            cat = r["failure_category"]
            pid = r["pid"]
            qd = r["query_date"]
            m = r["method"].upper()
            c = r["total_cost"]
            nl = r["n_llm_calls"]
            log_path = f"{active_out}/Run_Seed_{active_seed}/{r['method']}/{pid}_{qd}.json"
            origin = FAILURE_ORIGINS.get(cat, "Unclassified")
            print(f"  * [{cat}] {m} {pid} @ {qd} (cost={c}, llm={nl})")
            print(f"    -> Log File: {log_path}")
            print(f"       Grounding violations: {r['grounding_reasons']}")
            print(f"       Origin: {origin[:95]}...\n")

    print("=" * 70)
    print(f"All run artifacts and logs saved in: {run_dir}/")
    print("=" * 70)

if __name__ == "__main__":
    main()

Evaluating RULE on Seed 42 (24 questions)...
Evaluating RAG on Seed 42 (24 questions)...
Evaluating LLM on Seed 42 (24 questions)...
    [Network Notice] The read operation timed out. Retrying in 3.0s...

BENCHMARK SUMMARY TABLE (Seed: 42, 24 questions)
method      exact ground success abstain    cost  search    read budgetX     llm   malf  unauth unfaith       J
rule        1.000  1.000   1.000   0.000    8.46    2.46    3.00   0.000    0.00      0       0       0    0.85
rag         0.083  0.083   0.083   0.833    7.00    1.00    3.00   0.000    1.00      0      17       0    9.89
llm         0.750  0.750   0.750   0.000    8.25    1.00    3.62   0.000    6.21      0      14       0    3.45

FAILURE CATEGORIES BREAKDOWN
  - insufficient_evidence         : 20 episodes
  - failure_to_recognize_revision : 6 episodes
  - used_not_yet_effective_revision: 2 episodes

FAILED EPISODES LOG REGISTER (Detailed analysis injected into each file)
  * [insufficient_evidence] RAG P01 @ 2026-03-01 (c